In [1]:
import torch
import torch.nn as nn
from Settings import TinyStoriesLM
from datetime import datetime
from tokenizer import TinyStoriesTokenizer
from torch.utils.data import Dataset, DataLoader

print("cell ran")

cell ran


In [2]:
class ProbeDataset(Dataset):
    def __init__(self, data, limit=512, ignore_index=-100, pos_to_id=None):
        self.data = data
        self.limit = limit
        self.ignore_index = ignore_index
        self.pos_to_id = pos_to_id

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # to handle error of exeding block size
        input_ids = torch.tensor(item['input_ids'][:self.limit])
        
        # jsons didn't save the ids
        numeric_labels = []
        for tag in item['pos_tags'][:self.limit]:
            if tag == "IGNORE":
                numeric_labels.append(self.ignore_index)
            else:
                numeric_labels.append(self.pos_to_id[tag])
        
        labels = torch.tensor(numeric_labels)
        
        return {
            'input_ids': input_ids,
            'labels': labels
        }

print("cell ran")

cell ran


In [3]:
# We load the aligned data
import json

with open("aligned_data_metadata.json", "r") as f:
    metadata = json.load(f)
    pos_to_id = metadata["pos_to_id"]
    id_to_pos = metadata["id_to_pos"]
    ignore_index = metadata["ignore_index"]

with open("aligned_data.json", "r") as f:
    aligned_data = json.load(f)

print("cell ran")

cell ran


In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint_path = 'last_checkpoint_next_word.pt'

model = TinyStoriesLM.load(checkpoint_path, device=device).to(device)

# We have to freeze all the model
for param in model.parameters():
    param.requires_grad = False

# We set our model i eval mode -> deactivate dropout  so the vectors are stable
# dropout turns off random neurons during training to avoid overfitting/memorizing 
model.eval() #same input = same output

activations = {}

def forward_hook(module, input, output): # hook function
    # 'output' - ouput from Transformer layer
    # module layer that was just executed
    # save in our dictionary
    # .detach() no gradient info / history - we just want numbers
    
    activations[module.name] = output.detach() # save

hooks = []

# register each block (TransformerBlock)
for i, block in enumerate(model.transformers):
    block.name = f"layer_{i}" 
    handle = block.register_forward_hook(forward_hook)
    hooks.append(handle)


num_tags = len(pos_to_id)
input_dim = model.config.vector_dim
limit = model.config.block_size
num_layers = len(model.transformers)

# data for training
train_subset = aligned_data[:3500]

# data to validate results
val_subset = aligned_data[3500:4000]

# training
train_ds = ProbeDataset(train_subset, limit=limit, ignore_index=ignore_index, pos_to_id=pos_to_id)
train_loader = DataLoader(train_ds, batch_size=model.config.batch_size, shuffle=True)

# validation
val_ds = ProbeDataset(val_subset, limit=limit, ignore_index=ignore_index, pos_to_id=pos_to_id)
val_loader = DataLoader(val_ds, batch_size=model.config.batch_size, shuffle=False)

# Layer for layer
for layer_to_probe in range(num_layers):
    layer_name = f"layer_{layer_to_probe}"
    print(f"\nTraining probe of: {layer_name}")

    # we create probe for each layer
    probe = nn.Linear(input_dim, num_tags).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)

    # Training each probe
    for epoch in range(3):
        probe.train()
        
        for batch in train_loader:
            activations.clear()
            
            ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            with torch.no_grad():
                model(ids) # Forward pass del modelo para activar los hooks

            optimizer.zero_grad()
            logits = probe(activations[layer_name])
   
            loss = criterion(logits.view(-1, num_tags), labels.view(-1))
            loss.backward()
            optimizer.step()

        # we evaluate each epoch to see progress
        probe.eval()
        correct_tokens = 0
        total_tokens = 0
        
        with torch.no_grad():
            
            for v_batch in val_loader:
                v_ids = v_batch['input_ids'].to(device)
                v_labels = v_batch['labels'].to(device)
                
                model(v_ids)
                v_logits = probe(activations[layer_name])
                predictions = v_logits.argmax(dim=-1)

                # we don't want to evaluate tokens with ignore, just the one that have POS tags
                mask = (v_labels != ignore_index)
                correct_tokens += (predictions[mask] == v_labels[mask]).sum().item()
                total_tokens += mask.sum().item()
                            
        epoch_accuracy = (correct_tokens / total_tokens) * 100 if total_tokens > 0 else 0.0
        print(f"Epoch {epoch + 1} accuracy: {epoch_accuracy:.2f}%")


for handle in hooks:
    handle.remove()

# when trainning (ignore_index=-100) impo
print("cell ran")

Model loaded from last_checkpoint_next_word.pt (Epoch 0, iteration 40000)

Training probe of: layer_0
Epoch 1 accuracy: 15.95%
Epoch 2 accuracy: 15.88%
Epoch 3 accuracy: 15.96%

Training probe of: layer_1
Epoch 1 accuracy: 16.03%
Epoch 2 accuracy: 16.02%
Epoch 3 accuracy: 15.84%

Training probe of: layer_2
Epoch 1 accuracy: 15.80%
Epoch 2 accuracy: 15.78%
Epoch 3 accuracy: 15.93%

Training probe of: layer_3
Epoch 1 accuracy: 15.79%
Epoch 2 accuracy: 15.79%
Epoch 3 accuracy: 15.65%
cell ran
